In [3]:
import pandas as pd
from pathlib import Path

outdir = Path.cwd() / "output"

v1996 = pd.read_csv(outdir / "HCRIS_Data_v1996.txt", sep="\t")
v2010 = pd.read_csv(outdir / "HCRIS_Data_v2010.txt", sep="\t")

print(v1996.shape)
print(v2010.shape)

(78508, 47)
(57802, 48)


In [5]:
# combine both datasets
hcris = pd.concat([v1996, v2010], ignore_index=True)

print(hcris.shape)
hcris.head()

(136310, 53)


,report,provider_number,npi,fy_start,fy_end,date_processed,date_created,status,year,data_source,...,pps_ip_charges,pps_op_charges,pps_mcare_cost,pps_pgm_cost,hvbp_payment,hrrp_payment,tot_uncomp_care_charges,tot_uncomp_care_partial_pmts,bad_debt,depr_HIT
0,31,10007,NaN,10/01/1997,09/30/1998,12/11/2000,09/15/1999,2,1998,v1996,...,11363219.0,7770481.0,3343745.0,3016904.0,NaN,NaN,NaN,NaN,NaN,NaN
1,38,10008,NaN,07/01/1998,06/30/1999,12/11/2000,01/13/2000,1,1998,v1996,...,6668301.0,4690159.0,1548851.0,1447604.0,NaN,NaN,NaN,NaN,NaN,NaN
2,50,10010,NaN,10/01/1997,09/30/1998,12/11/2000,06/30/1999,2,1998,v1996,...,32315430.0,24962516.0,8527466.0,7439820.0,NaN,NaN,NaN,NaN,NaN,NaN
3,69,10015,NaN,10/01/1997,03/31/1998,11/14/2000,06/02/1999,2,1998,v1996,...,2384290.0,2215334.0,656756.0,568522.0,NaN,NaN,NaN,NaN,NaN,NaN
4,97,10021,NaN,10/01/1997,09/30/1998,12/05/2000,09/28/1999,2,1998,v1996,...,21843543.0,16168437.0,5042593.0,4719170.0,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# convert dates
hcris['fy_end'] = pd.to_datetime(hcris['fy_end'], errors='coerce')

# create fiscal year
hcris['fyear'] = hcris['fy_end'].dt.year

In [8]:
hcris["fy_start"] = pd.to_datetime(hcris["fy_start"], errors="coerce")
hcris["fy_end"] = pd.to_datetime(hcris["fy_end"], errors="coerce")

In [9]:
# count reports per hospital-fyear
hcris['total_reports'] = hcris.groupby(['provider_number', 'fyear'])['provider_number'].transform('count')
hcris['report_number'] = hcris.groupby(['provider_number', 'fyear']).cumcount() + 1

# --- Set 1: hospitals with only one report ---
unique1 = hcris[hcris['total_reports'] == 1].copy()
unique1 = unique1.drop(columns=['report', 'total_reports', 'report_number', 'npi', 'status'])
unique1['source'] = 'unique reports'

# --- Duplicates: hospitals with multiple reports ---
dupes = hcris[hcris['total_reports'] > 1].copy()
dupes['time_diff'] = (dupes['fy_end'] - dupes['fy_start']).dt.days
dupes['total_days'] = dupes.groupby(['provider_number', 'fyear'])['time_diff'].transform('sum')

In [10]:
# --- Set 2: multiple reports summing to <370 days -> aggregate ---
dupes_short = dupes[dupes['total_days'] < 370].copy()
dupes_short['hrrp_payment'] = dupes_short['hrrp_payment'].fillna(0)
dupes_short['hvbp_payment'] = dupes_short['hvbp_payment'].fillna(0)

sum_cols = ['tot_charges', 'tot_discounts', 'tot_operating_exp', 'ip_charges',
            'icu_charges', 'ancillary_charges', 'tot_discharges', 'mcare_discharges',
            'mcaid_discharges', 'tot_mcare_payment', 'secondary_mcare_payment',
            'hvbp_payment', 'hrrp_payment']

agg_dict = {c: 'sum' for c in sum_cols}
agg_dict['beds'] = 'max'
agg_dict['fy_start'] = 'min'
agg_dict['fy_end'] = 'max'
agg_dict['date_processed'] = 'max'
agg_dict['date_created'] = 'min'
agg_dict['street'] = 'first'
agg_dict['city'] = 'first'
agg_dict['state'] = 'first'
agg_dict['zip'] = 'first'
agg_dict['county'] = 'first'

unique2 = dupes_short.groupby(['provider_number', 'fyear'], as_index=False).agg(agg_dict)
unique2['source'] = 'total for year'

In [12]:
num_cols = [
    "beds", "tot_charges", "tot_discounts", "net_pat_rev", "tot_operating_exp",
    "ip_charges", "icu_charges", "ancillary_charges", "tot_discharges",
    "mcare_discharges", "mcaid_discharges", "tot_mcare_payment",
    "secondary_mcare_payment", "hvbp_payment", "hrrp_payment",
    "tot_uncomp_care_charges", "tot_uncomp_care_partial_pmts", "bad_debt",
    "cost_to_charge", "new_cap_ass", "cash", "fixed_assets", "depr_land",
    "depr_bldg", "depr_lease", "depr_fixed_equip", "depr_auto",
    "depr_major_equip", "depr_minor_equip", "depr_HIT",
    "current_assets", "current_liabilities", "time_diff", "total_days"
]

for c in num_cols:
    if c in dupes_remain.columns:
        dupes_remain[c] = pd.to_numeric(dupes_remain[c], errors="coerce")

In [13]:
date_cols = ["fy_start", "fy_end", "date_processed", "date_created"]
for c in date_cols:
    if c in dupes_remain.columns:
        dupes_remain[c] = pd.to_datetime(dupes_remain[c], errors="coerce")

In [16]:
# convert date columns
for c in ["date_processed", "date_created", "fy_start", "fy_end"]:
    if c in dupes_remain.columns:
        dupes_remain[c] = pd.to_datetime(dupes_remain[c], errors="coerce")

# numeric columns
num_cols = [
    "beds", "tot_charges", "tot_discounts", "net_pat_rev", "tot_operating_exp",
    "ip_charges", "icu_charges", "ancillary_charges", "tot_discharges",
    "mcare_discharges", "mcaid_discharges", "tot_mcare_payment",
    "secondary_mcare_payment", "hvbp_payment", "hrrp_payment",
    "tot_uncomp_care_charges", "tot_uncomp_care_partial_pmts", "bad_debt",
    "cost_to_charge", "new_cap_ass", "cash", "fixed_assets", "depr_land",
    "depr_bldg", "depr_lease", "depr_fixed_equip", "depr_auto",
    "depr_major_equip", "depr_minor_equip", "depr_HIT",
    "current_assets", "current_liabilities", "time_diff", "total_days"
]

for c in num_cols:
    if c in dupes_remain.columns:
        dupes_remain[c] = pd.to_numeric(dupes_remain[c], errors="coerce")

In [20]:
# --- FIX TYPES BEFORE AGG ---

# convert dates
date_cols = ["fy_start", "fy_end", "date_processed", "date_created"]
for c in date_cols:
    if c in dupes_remain.columns:
        dupes_remain[c] = pd.to_datetime(dupes_remain[c], errors="coerce")

# convert numerics
num_cols = dupes_remain.columns.difference([
    "provider_number", "npi", "street", "city", "state", "zip",
    "county", "name", "data_source", "status"
])

for c in num_cols:
    if dupes_remain[c].dtype == "object":
        dupes_remain[c] = pd.to_numeric(dupes_remain[c], errors="ignore")

In [21]:
agg_dict = {
    # IDs / text → first
    "report": "first",
    "npi": "first",
    "status": "first",
    "data_source": "first",
    "street": "first",
    "city": "first",
    "state": "first",
    "zip": "first",
    "county": "first",
    "name": "first",

    # dates → safe now
    "fy_start": "min",
    "fy_end": "max",
    "date_processed": "max",
    "date_created": "max",

    # numerics → average
    "beds": "mean",
    "tot_charges": "mean",
    "tot_discounts": "mean",
    "net_pat_rev": "mean",
    "tot_operating_exp": "mean",
    "ip_charges": "mean",
    "icu_charges": "mean",
    "ancillary_charges": "mean",
    "tot_discharges": "mean",
    "mcare_discharges": "mean",
    "mcaid_discharges": "mean",
    "tot_mcare_payment": "mean",
    "secondary_mcare_payment": "mean",
    "hvbp_payment": "mean",
    "hrrp_payment": "mean",

    # helper vars
    "time_diff": "sum",
    "total_days": "first",
    "total_reports": "first",
    "report_number": "first"
}

In [22]:
unique4 = dupes_remain.groupby(["provider_number", "fyear"], as_index=False).agg(agg_dict)
unique4["source"] = "weighted_average"

In [24]:
# make sure dates are real datetimes
for c in ['fy_start', 'fy_end', 'date_processed', 'date_created']:
    if c in dupes_remain.columns:
        dupes_remain[c] = pd.to_datetime(dupes_remain[c], errors='coerce')

# make sure weighted/numeric columns are numeric
num_cols = [
    'beds', 'tot_charges', 'tot_discounts', 'net_pat_rev', 'tot_operating_exp',
    'ip_charges', 'icu_charges', 'ancillary_charges', 'tot_discharges',
    'mcare_discharges', 'mcaid_discharges', 'tot_mcare_payment',
    'secondary_mcare_payment', 'hvbp_payment', 'hrrp_payment',
    'tot_uncomp_care_charges', 'tot_uncomp_care_partial_pmts', 'bad_debt',
    'cost_to_charge', 'new_cap_ass', 'cash', 'fixed_assets', 'depr_land',
    'depr_bldg', 'depr_lease', 'depr_fixed_equip', 'depr_auto',
    'depr_major_equip', 'depr_minor_equip', 'depr_HIT',
    'current_assets', 'current_liabilities', 'time_diff', 'total_days',
    'total_reports', 'report_number'
]

for c in num_cols:
    if c in dupes_remain.columns:
        dupes_remain[c] = pd.to_numeric(dupes_remain[c], errors='coerce')

In [25]:
agg_dict = {
    'report': 'first',
    'npi': 'first',
    'fy_start': 'min',
    'fy_end': 'max',
    'date_processed': 'max',
    'date_created': 'max',
    'status': 'first',
    'data_source': 'first',
    'street': 'first',
    'city': 'first',
    'state': 'first',
    'zip': 'first',
    'county': 'first',
    'name': 'first',
    'beds': 'mean',
    'tot_charges': 'sum',
    'tot_discounts': 'sum',
    'net_pat_rev': 'mean',
    'tot_operating_exp': 'sum',
    'ip_charges': 'sum',
    'icu_charges': 'sum',
    'ancillary_charges': 'sum',
    'tot_discharges': 'sum',
    'mcare_discharges': 'sum',
    'mcaid_discharges': 'sum',
    'tot_mcare_payment': 'sum',
    'secondary_mcare_payment': 'sum',
    'hvbp_payment': 'sum',
    'hrrp_payment': 'sum',
    'tot_uncomp_care_charges': 'mean',
    'tot_uncomp_care_partial_pmts': 'mean',
    'bad_debt': 'mean',
    'cost_to_charge': 'mean',
    'new_cap_ass': 'mean',
    'cash': 'mean',
    'fixed_assets': 'mean',
    'depr_land': 'mean',
    'depr_bldg': 'mean',
    'depr_lease': 'mean',
    'depr_fixed_equip': 'mean',
    'depr_auto': 'mean',
    'depr_major_equip': 'mean',
    'depr_minor_equip': 'mean',
    'depr_HIT': 'mean',
    'current_assets': 'mean',
    'current_liabilities': 'mean',
    'time_diff': 'sum',
    'total_days': 'first',
    'total_reports': 'first',
    'report_number': 'first',
    'max_days': 'first',
    'max_date': 'first'
}

In [27]:
# Set 4: remaining -> weighted average by days
dupes_remain = dupes_long.merge(
    unique3[['provider_number', 'fyear']].drop_duplicates(),
    on=['provider_number', 'fyear'], how='left', indicator=True
)
dupes_remain = dupes_remain[dupes_remain['_merge'] == 'left_only'].drop(columns=['_merge']).copy()

weight_cols = ['tot_charges', 'tot_discounts', 'tot_operating_exp', 'ip_charges',
               'icu_charges', 'ancillary_charges', 'tot_discharges', 'mcare_discharges',
               'mcaid_discharges', 'tot_mcare_payment', 'secondary_mcare_payment',
               'hvbp_payment', 'hrrp_payment']

for c in weight_cols:
    dupes_remain[c] = pd.to_numeric(dupes_remain[c], errors='coerce')

dupes_remain['time_diff'] = pd.to_numeric(dupes_remain['time_diff'], errors='coerce')
dupes_remain['total_days'] = pd.to_numeric(dupes_remain['total_days'], errors='coerce')

dupes_remain['hrrp_payment'] = dupes_remain['hrrp_payment'].fillna(0)
dupes_remain['hvbp_payment'] = dupes_remain['hvbp_payment'].fillna(0)

for c in weight_cols:
    dupes_remain[c] = dupes_remain[c] * (dupes_remain['time_diff'] / dupes_remain['total_days'])

for c in ['fy_start', 'fy_end', 'date_processed', 'date_created']:
    if c in dupes_remain.columns:
        dupes_remain[c] = pd.to_datetime(dupes_remain[c], errors='coerce')

unique4 = dupes_remain.groupby(['provider_number', 'fyear'], as_index=False).agg(agg_dict)
unique4['source'] = 'weighted_average'

In [29]:
final = pd.concat([unique1, unique2, unique3, unique4], ignore_index=True)

# if both exist, keep fyear as the final year and drop old year
if 'year' in final.columns and 'fyear' in final.columns:
    final = final.drop(columns=['year'])

final = final.rename(columns={'fyear': 'year'})
final = final.sort_values(['provider_number', 'year']).reset_index(drop=True)

In [30]:
print(final.columns[final.columns.duplicated()].tolist())

[]


In [31]:
from pathlib import Path

outdir = Path.cwd() / "output" / "cleaned"
outdir.mkdir(parents=True, exist_ok=True)

for y in sorted(final['year'].dropna().unique()):
    y = int(y)
    final[final['year'] == y].to_csv(outdir / f"data-{y}.csv", index=False)

print("Saved files to:", outdir)

Saved files to: /home/gdanzil/econ470/a0/work/HW4/output/cleaned
